# LeetCode 20 — Valid Parentheses | Nhập môn Python, giảng từng bước

**Độ khó:** Easy · **Kỹ thuật trọng tâm:** Stack · LIFO · khớp cặp  
**Bài gốc:** https://leetcode.com/problems/valid-parentheses/

Notebook này dành cho người **mới bắt đầu**. Đọc các ô từ trên xuống: trước hết hiểu **vì sao**, sau đó chạy code nhỏ để tự thấy **cách làm**.

**Khung hàm (signature) cần nộp LeetCode:**

```python
class Solution:
    def isValid(self, s: str) -> bool:
        ...
```

**Sau bài học, bạn có thể:**
- Phân biệt đủ số cặp với lồng cặp đúng thứ tự.
- Tự thực hành list như stack với append và pop.
- Cài thuật toán O(n), thử edge cases và giải thích invariant.

**Cách sử dụng:** chạy từng code cell theo thứ tự bằng `Shift + Enter`. Những ô có `TODO` là bài tự luyện: chỉ định nghĩa hàm hoặc chứa lời nhắc kiểm thử, **không được gọi tự động** khi chưa làm xong.

> Lộ trình: hiểu đề → Python căn bản → cách trực giác → phát hiện nút thắt → cải tiến → trace → nộp LeetCode → test → lỗi → vận dụng → bài tập → đáp án → checklist.

## 1. Hiểu đề bằng các phản ví dụ

Chuỗi `s` chỉ gồm `(`, `)`, `[`, `]`, `{`, `}`. Trả `True` nếu **mọi dấu mở** được đóng bởi **đúng loại dấu đóng** và theo đúng **thứ tự lồng nhau**.

| s | hợp lệ | giải thích |
|---|---|---|
| `"()"` | True | mở `(`, đóng `)` |
| `"()[]{}"` | True | 3 cặp cạnh nhau |
| `"(]"` | False | sai loại |
| `"([)]"` | False | sai thứ tự lồng |
| `"{[]}"` | True | đóng từ trong ra ngoài |
| `"("` | False | còn dấu mở chưa đóng |

**Lưu ý:** đếm số dấu mở và dấu đóng bằng nhau **chưa đủ**: `"([)]"` có số lượng bằng nhau nhưng sai thứ tự.

In [ ]:
samples = ["()", "()[]{}", "(]", "([)]", "{[]}", "("]
for s in samples:
    print(f"{s!r} -> cần kiểm tra đúng cặp và thứ tự, không chỉ đếm số lượng")

## 2. Vấn đề thứ tự — tưởng tượng một chồng đĩa

Khi vừa thấy dấu mở `{`, sau đó `[`, dấu được phép đóng **ngay tiếp theo** là `]`, **không phải** `}`. Dấu mở mới nhất phải đóng trước. Đây là quy tắc **LIFO: Last In, First Out** (vào sau, ra trước).

Cấu trúc dữ liệu giữ đúng quy tắc đó gọi là **stack (ngăn xếp)**. Hình dung xếp đĩa: chỉ lấy được đĩa trên cùng.

## 3. Python căn bản — list làm stack

Trong Python, `stack = []` tạo list rỗng; `.append(x)` **thêm** `x` lên đỉnh; `.pop()` **lấy ra và xóa** phần tử đỉnh (cuối list). `stack[-1]` xem phần tử đỉnh **không xóa**.

`pop()` trên list rỗng gây `IndexError`, vì vậy kiểm tra `if not stack` trước.

In [ ]:
stack = []
stack.append("("); print("Sau mở (:", stack)
stack.append("["); print("Sau mở [:", stack)
print("Đỉnh:", stack[-1])
print("Đóng đỉnh:", stack.pop())
print("Đỉnh mới:", stack[-1]); print("Còn lại:", stack)

## 4. Bảng ánh xạ cặp đóng → cặp mở

Một **dictionary (`dict`)** lưu cặp `key: value`. Với `pairs = {')':'(', ']':'[', '}':'{'}`, ta tra `pairs[ch]` để biết **dấu mở cần có** khi gặp dấu đóng `ch`.

`ch in pairs` kiểm tra `ch` có phải ký tự **đóng**; nếu không thì theo điều kiện đề, đó là ký tự **mở**.

In [ ]:
pairs = {")": "(", "]": "[", "}": "{"}
print("Dấu ) cần mở:", pairs[")"])
print("Dấu ] cần mở:", pairs["]"])
print("'(' là dấu đóng?", "(" in pairs)
print("'}' là dấu đóng?", "}" in pairs)

## 5. Cách trực giác — liên tục xóa các cặp liền nhau

Không cần stack, ta có thể tìm `"()"`, `"[]"`, `"{}"` rồi xóa. Lặp đến khi không thể xóa nữa. Nếu còn ký tự, chuỗi sai.

Ví dụ `"{[]}"` → `"{}"` → `""` ⇒ hợp lệ. Tuy nhiên cách này **duyệt lại chuỗi nhiều lần** và tạo chuỗi mới, tốn O(n²) trong trường hợp lồng sâu.

In [ ]:
def is_valid_by_replacing(s: str) -> bool:
    previous = None
    while s != previous:
        previous = s
        for pair in ["()", "[]", "{}"]:
            s = s.replace(pair, "")
    return s == ""

for s in ["{[]}", "([)]", "()[]{}", "(((" ]:
    print(repr(s), is_valid_by_replacing(s))

## 6. Insight tối ưu: giữ đúng những dấu *chưa đóng*

Khi thấy dấu **mở**, cất lên stack. Khi thấy dấu **đóng**, bắt buộc đỉnh stack phải đúng dấu mở của nó; nếu không thì trả `False` **ngay lập tức**. Đúng cặp thì `pop()` lấy ra.

Sau khi duyệt hết, stack **phải rỗng**: rỗng nghĩa là mọi dấu mở đã có dấu đóng. Không rỗng nghĩa là còn nợ dấu đóng.

## 7. Trace tính tay cho `{[()]}`

| bước | ký tự | stack trước | hành động | stack sau |
|---:|:---:|:---|:---|:---|
| 0 | `{` | `[]` | thêm `{` | `['{']` |
| 1 | `[` | `['{']` | thêm `[` | `['{','[']` |
| 2 | `(` | `['{','[']` | thêm `(` | `['{','[','(']` |
| 3 | `)` | `['{','[','(']` | pop đúng `(` | `['{','[']` |
| 4 | `]` | `['{','[']` | pop đúng `[` | `['{']` |
| 5 | `}` | `['{']` | pop đúng `{` | `[]` |

Duyệt xong stack rỗng ⇒ `True`. Nếu đổi ký tự thứ 3 từ `(` thành `]`, đỉnh đang là `[`? hãy tự chạy trace.

## 8. Code chính — mỗi ký tự xử lý 1 lần

**Đừng đặt `pop` trước khi kiểm tra stack rỗng**. Ta kiểm tra `not stack` hoặc `stack[-1] != pairs[ch]`, sau đó mới lấy dấu mở ra khỏi stack.

In [ ]:
def valid_parentheses_stack(s: str) -> bool:
    pairs = {")": "(", "]": "[", "}": "{"}
    stack = []
    for ch in s:
        if ch not in pairs:
            stack.append(ch)
        else:
            if not stack or stack[-1] != pairs[ch]:
                return False
            stack.pop()
    return len(stack) == 0

for s in ["()", "([)]", "{[]}", "("]:
    print(s, "->", valid_parentheses_stack(s))

## 9. Trace chạy thật — trước và sau cập nhật

Hàm này in rõ từng hành động; input `"([)]"` sẽ **dừng ở chỗ sai**, không cần duyệt hết. Đó chính là lợi thế của việc kiểm tra khi gặp dấu đóng.

In [ ]:
def trace_parentheses(s):
    pairs = {")": "(", "]": "[", "}": "{"}
    stack = []
    print("Chuỗi:", repr(s))
    for i, ch in enumerate(s):
        print(f"i={i}, ch={ch!r}, trước={stack}", end=" ")
        if ch not in pairs:
            stack.append(ch)
            print("-> PUSH; sau=", stack)
        elif not stack or stack[-1] != pairs[ch]:
            print("-> SAI: thiếu/sai dấu mở; STOP")
            return False
        else:
            popped = stack.pop()
            print(f"-> POP {popped!r}; sau={stack}")
    print("Kết thúc; stack còn:", stack)
    return not stack

assert trace_parentheses("{[()]}") is True
assert trace_parentheses("([)]") is False

## 10. Code chuẩn để nộp LeetCode

Hàm submit không cần `print`. Vẫn giữ nguyên sự phân biệt “dấu mở thêm vào” / “dấu đóng so với đỉnh”.

In [ ]:
class Solution:
    def isValid(self, s: str) -> bool:
        pairs = {")": "(", "]": "[", "}": "{"}
        stack = []
        for ch in s:
            if ch not in pairs:
                stack.append(ch)
            else:
                if not stack or stack[-1] != pairs[ch]:
                    return False
                stack.pop()
        return not stack

print("Submit demo:", Solution().isValid("{[]}"))

## 11. Vì sao stack giải đúng bài?

**Bất biến:** sau khi xử lý các ký tự đầu, stack lưu **đúng những dấu mở đã gặp nhưng chưa được đóng**, theo thứ tự đi từ ngoài vào trong.

- Gặp dấu mở: thêm nó vào stack, vẫn giữ bất biến.
- Gặp dấu đóng: chỉ có dấu mở mới nhất được phép ghép; không khớp thì mọi cách đóng sau cũng không sửa được sai lệch hiện tại.
- Hết chuỗi: chỉ hợp lệ khi không còn dấu mở chưa đóng (`not stack`).

**Độ phức tạp:** `n = len(s)`. Mỗi ký tự đi qua đúng một lần, mỗi `append`/`pop` là O(1) ⇒ **O(n)** thời gian. Tối đa có `n` dấu mở trong stack ⇒ **O(n)** bộ nhớ phụ.

## 12. Test kỹ các trường hợp dễ nhầm

Ta kiểm tra chuỗi có các cặp độc lập, cặp lồng nhau, sai loại, sai thứ tự, dư mở, dư đóng. Ví dụ chuỗi rỗng không nằm trong constraints thông thường, nhưng thuật toán cũng trả True như một **mở rộng hợp lý**.

In [ ]:
tests_20 = [
    ("()", True), ("()[]{}", True), ("(]", False),
    ("([)]", False), ("{[]}", True), ("(", False),
    ("]", False), ("(())", True), ("[({})]", True),
    ("([", False), ("([{}])", True), ("", True),
]
for s, expected in tests_20:
    actual = Solution().isValid(s)
    assert actual == expected, (s, actual, expected)
    assert is_valid_by_replacing(s) == actual
print("PASS", len(tests_20), "ca kiểm thử + đối chiếu cách replace")

## 13. Sai lầm phổ biến — nhìn code sai để hiểu

1. **Chỉ đếm**: với `"([)]"`, số dấu mở và đóng bằng nhau nhưng thứ tự sai.
2. **`pop()` khi rỗng**: `s=")"` làm chương trình lỗi thay vì `False`.
3. **Đảo ngược cặp**: gặp `")"` phải tìm `"("`, không phải `"]"` hay `")"`.
4. **Trả `True` ngay khi duyệt xong**: với `"(("` stack còn `['(', '(']`, vậy phải `False`.
5. **Dùng queue thay stack**: dấu mở đầu không phải luôn đóng đầu tiên trong cấu trúc lồng.

In [ ]:
s = "([)]"
opens = sum(ch in "([{ " for ch in s)
closes = sum(ch in ")] }".replace(" ", "") for ch in s)
print("Đếm đơn thuần:", opens, closes, "| thực tế:", valid_parentheses_stack(s))

## 14. Khi nào nên nghĩ tới stack?

Khi bài toán có yếu tố **cấu trúc lồng nhau**, **đóng cái gần nhất**, **undo**, **quay lại trạng thái gần nhất**, **ngoặc**, hãy nghĩ tới stack. Kỹ thuật này cũng xuất hiện trong kiểm tra biểu thức, parser, lịch sử thao tác và DFS.

## 15. Bài tập (5 mức) — tự làm trước khi nhìn phần giải

### Bài 1 — Một cặp ngoặc — Tự làm

Viết hàm `is_one_pair` chỉ trả True cho đúng một cặp như `"()"`, `"[]"`, `"{}"`.

**Gợi ý cấp 1 (tư duy):** Chuỗi chỉ dài 2 ký tự.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

So sánh với 3 chuỗi bằng `in`.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

`return s in (...)`.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_20_1(s: str) -> bool:
    # TODO: trả True khi s là một trong ba cặp hợp lệ
    pass

# assert exercise_20_1("[]") is True
# assert exercise_20_1("(]") is False

### Bài 2 — Trace tính tay — Tự làm

Cho `s="([{}])"`, hãy viết **dãy số phần tử stack** sau mỗi ký tự.

**Gợi ý cấp 1 (tư duy):** Mở: cộng 1, đóng đúng: trừ 1.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

Lần lượt xét 6 ký tự.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

Bắt đầu 0; ba dấu mở thì 1,2,3; ba dấu đóng thì 2,1,0.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_20_2():
    # TODO: trả về list gồm 6 kích thước stack sau mỗi ký tự
    pass

# assert exercise_20_2() == [1, 2, 3, 2, 1, 0]

### Bài 3 — Điền stack chính — Tự làm

Viết lại hàm `valid` dùng `append` và `pop`.

**Gợi ý cấp 1 (tư duy):** LIFO: dấu mới nhất đóng trước.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

`ch not in pairs`, `stack[-1]`, `stack.pop()`.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

Kiểm tra `not stack or stack[-1] != pairs[ch]` trước khi pop.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_20_3(s: str) -> bool:
    # TODO: dictionary pairs
    # TODO: stack = []
    # TODO: duyệt từng ký tự, push khi mở
    # TODO: khi đóng, so đỉnh rồi pop
    pass

# assert exercise_20_3("([])") is True
# assert exercise_20_3("([)]") is False

### Bài 4 — Đếm số dấu mở chưa đóng — Tự làm

Biến thể: với chuỗi **chỉ chứa dấu mở** `([{`, viết hàm trả độ sâu stack sau khi đọc hết.

**Gợi ý cấp 1 (tư duy):** Mỗi ký tự đều mở nên đều push.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

Dùng `.append(ch)` trong `for`.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

Trả `len(stack)`.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_20_4(s: str) -> int:
    # TODO: dùng một list stack và len(stack)
    pass

# assert exercise_20_4("([{") == 3

### Bài 5 — Mini challenge lấy độ sâu lớn nhất — Tự làm

Với chuỗi hợp lệ chỉ gồm `(` và `)`, trả mức lồng sâu nhất. Ví dụ `"(()())"` -> 2. Đây là **bài biến thể**.

**Gợi ý cấp 1 (tư duy):** Độ sâu tăng khi mở, giảm khi đóng.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

Biến `depth`, `best`; dùng `max`.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

Nếu `ch == "("`: `depth += 1`, `best = max(best, depth)`; else giảm.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_20_5(s: str) -> int:
    # TODO: duyệt, tăng/decrease độ sâu, nhớ max
    pass

# assert exercise_20_5("(()())") == 2

## 16. Đáp án tham khảo bài tập

In [ ]:
def answer_20_1(s):
    return s in ("()", "[]", "{}")

def answer_20_2():
    return [1, 2, 3, 2, 1, 0]

def answer_20_3(s):
    pairs = {")": "(", "]": "[", "}": "{"}
    stack = []
    for ch in s:
        if ch not in pairs:
            stack.append(ch)
        else:
            if not stack or stack[-1] != pairs[ch]:
                return False
            stack.pop()
    return not stack

def answer_20_4(s):
    stack = []
    for ch in s:
        stack.append(ch)
    return len(stack)

def answer_20_5(s):
    depth = best = 0
    for ch in s:
        if ch == "(":
            depth += 1
            best = max(best, depth)
        else:
            depth -= 1
    return best

assert answer_20_1("[]")
assert answer_20_2() == [1, 2, 3, 2, 1, 0]
assert answer_20_3("([)]") is False
assert answer_20_4("([{") == 3
assert answer_20_5("(()())") == 2
print("Đáp án tham khảo: PASS 5/5")

## Tóm tắt — mẫu tư duy cần nhớ

**Mẫu Stack:** gặp mở thì `append`; gặp đóng thì bắt buộc đỉnh tương ứng và `pop`; hết chuỗi kiểm tra stack rỗng. **O(n)** thời gian, **O(n)** bộ nhớ.

## Checklist trước khi submit

- [ ] Tôi phân biệt được đúng số cặp với đúng thứ tự.
- [ ] Tôi biết `pop()` trên stack rỗng gây lỗi.
- [ ] Tôi hiểu `not stack` tại sao cần kiểm tra cuối cùng.
- [ ] Tôi tự mô phỏng được `([)]` và `{[]}`.
- [ ] Tôi chỉ dán class Solution khi submit.

**Tự kiểm tra lần cuối:** có thể giải thích lời giải bằng lời trước khi xem code? Có tự tính được ít nhất một bảng trace? Nếu chưa, quay lại phần mô phỏng.

*Notebook học tập được biên soạn theo cẩm nang tiếng Việt đính kèm; tiêu đề/đề/ràng buộc bám sát LeetCode. Những biến thể được đánh dấu là mở rộng, không phải yêu cầu đề chính.*